# qwen3-vl:8b-instruct Ollama server cho GUI Agent (OOP_Project_2026)

Notebook này dùng để:
1. Cài Ollama trên Kaggle GPU (T4 x2, free tier)
2. Pull `qwen3-vl:8b-instruct` (hoặc load lại từ Dataset đã cache)
3. Expose Ollama server ra ngoài qua Cloudflare quick tunnel
4. In ra URL public để paste vào `.env` của project local

**Trước khi chạy:** Settings (góc phải) -> Accelerator = GPU T4 x2, Internet = ON.

In [1]:
# Cell 0 - Cai zstd (bat buoc, ollama install.sh moi can cai nay de giai nen)
!apt-get update -qq && apt-get install -y -qq zstd
print("zstd installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
zstd installed.


In [2]:
# Cell 1 - Cai Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed.")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                   32.1%############################################                    76.3%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installed.


In [3]:
# Cell 2 - Start Ollama server nen
import subprocess, time, os

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"  # bat buoc de tunnel forward duoc

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,  # tach process group rieng - khong bi Kernel>Interrupt/Stop giet theo
)
time.sleep(6)
print("Ollama server PID:", ollama_proc.pid)

Ollama server PID: 985


In [4]:
# Cell 3 - Pull model (chi can chay 1 lan / lan dau moi session neu chua cache)
# Neu ban da dong goi ~/.ollama/models thanh Kaggle Dataset va Add Data vao notebook nay,
# hay copy du lieu vao dung path TRUOC khi chay cell nay de tranh tai lai:
#
#   !mkdir -p ~/.ollama/models
#   !cp -r /kaggle/input/<ten-dataset-cache>/* ~/.ollama/models/
#
!ollama pull qwen3-vl:8b-instruct

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 1329cd5ab37e:   0% ▕                  ▏ 2.9 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   1% ▕                  ▏  86 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   2% ▕                  ▏ 124 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   4% ▕                  ▏ 214 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   5% ▕                  ▏ 301 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   6% ▕█                 ▏ 395 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   7% ▕█                 ▏ 442 MB/6.1 GB                  pulling manifest 
pulling 1329cd5ab37e:   9% ▕█

In [5]:
# Cell 4 - Test nhanh model da load dung chua (text-only ping)
!curl -s http://localhost:11434/api/generate -d '{\n  "model": "qwen3-vl:8b",\n  "prompt": "Reply with only the word OK",\n  "stream": false\n}'

{"error":"invalid character '\\\\' looking for beginning of object key string"}

In [6]:
# Cell 5 - Cai cloudflared va mo tunnel
!curl -Lo cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("cloudflared ready.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 37.9M  100 37.9M    0     0  26.1M      0  0:00:01  0:00:01 --:--:--  101M
cloudflared ready.


In [16]:
# Cell 6 - Chay tunnel, tu dong bat URL public tu output
import subprocess, re, threading

tunnel_url = {"value": None}

def run_tunnel():
    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,  # tach process group rieng - khong bi Kernel>Interrupt/Stop giet theo
    )
    for line in proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m and tunnel_url["value"] is None:
            tunnel_url["value"] = m.group(0)
            print("\n=== PUBLIC OLLAMA ENDPOINT ===")
            print(tunnel_url["value"])
            print("===============================\n")
            print("Paste URL nay vao .env (OLLAMA_HOST=...) o may local, hoac chay:")
            print(f"  python update_ollama_env.py {tunnel_url['value']}")

thread = threading.Thread(target=run_tunnel, daemon=True)
thread.start()


=== PUBLIC OLLAMA ENDPOINT ===
https://solve-indirect-wells-delaware.trycloudflare.com

Paste URL nay vao .env (OLLAMA_HOST=...) o may local, hoac chay:
  python update_ollama_env.py https://solve-indirect-wells-delaware.trycloudflare.com


## Giu session song
Chay cell duoi day 1 lan. No chay NGAM O BACKGROUND THREAD (khong chan cell,
khong can Interrupt) - cu 5 phut in 1 dong timestamp de Kaggle khong coi la
idle. Ban van mo cell moi, chay !ollama list, v.v. binh thuong ma khong lam
chet ollama serve / cloudflared nua.

**Nho theo doi quota 30h/tuan o Settings > Accelerator.**

In [17]:
# Cell 7 - Keep-alive (chay NGAM o background thread, KHONG can Interrupt)
import time, datetime, threading

def _keep_alive():
    while True:
        print("alive at", datetime.datetime.now().isoformat())
        time.sleep(300)

threading.Thread(target=_keep_alive, daemon=True).start()
print("Keep-alive thread started - cu 5 phut se tu in 1 dong, khong can lam gi them.")

alive at 2026-08-13T11:09:32.716286
Keep-alive thread started - cu 5 phut se tu in 1 dong, khong can lam gi them.


In [18]:
!ollama list

]11;?\NAME                    ID              SIZE      MODIFIED      
qwen3-vl:8b-instruct    0533d74300e4    6.1 GB    6 minutes ago    


In [19]:
import subprocess, time

for i in range(2):
    t0 = time.time()
    result = subprocess.run(
        ["curl", "-s", "http://localhost:11434/api/generate", "-d",
         '{"model": "qwen3-vl:8b-instruct", "prompt": "Reply with only the word OK", "stream": false}'],
        capture_output=True, text=True
    )
    print(f"--- Lan {i+1}: {time.time() - t0:.2f} giay ---")
    print(result.stdout)
    print()

--- Lan 1: 0.47 giay ---
{"model":"qwen3-vl:8b-instruct","created_at":"2026-08-13T11:09:40.272575169Z","response":"OK","done":true,"done_reason":"stop","context":[151644,872,198,20841,448,1172,279,3409,10402,151645,198,151644,77091,198,3925],"total_duration":453028456,"load_duration":370951768,"prompt_eval_count":14,"prompt_eval_duration":38921000,"eval_count":2,"eval_duration":40454000}

--- Lan 2: 0.45 giay ---
{"model":"qwen3-vl:8b-instruct","created_at":"2026-08-13T11:09:40.718420425Z","response":"OK","done":true,"done_reason":"stop","context":[151644,872,198,20841,448,1172,279,3409,10402,151645,198,151644,77091,198,3925],"total_duration":435947338,"load_duration":359846776,"prompt_eval_count":14,"prompt_eval_duration":36819000,"eval_count":2,"eval_duration":36507000}



In [20]:
import subprocess, time
t0 = time.time()
result = subprocess.run(
    ["curl", "-s", "http://localhost:11434/api/generate", "-d",
     '{"model": "qwen3-vl:8b-instruct", "prompt": "Reply with only the word OK", "stream": false}'],
    capture_output=True, text=True
)
print("Thoi gian:", time.time() - t0, "giay")
print(result.stdout)

Thoi gian: 0.4482851028442383 giay
{"model":"qwen3-vl:8b-instruct","created_at":"2026-08-13T11:09:41.171933977Z","response":"OK","done":true,"done_reason":"stop","context":[151644,872,198,20841,448,1172,279,3409,10402,151645,198,151644,77091,198,3925],"total_duration":436967381,"load_duration":360510662,"prompt_eval_count":14,"prompt_eval_duration":36807000,"eval_count":2,"eval_duration":36963000}


In [21]:
!ollama ps

]11;?\NAME                    ID              SIZE     PROCESSOR    CONTEXT    UNTIL              
qwen3-vl:8b-instruct    0533d74300e4    10 GB    100% GPU     32768      4 minutes from now    
